In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
bbknn_epithelial_subtype_analysis.ipynb

In-depth analysis of specific epithelial cell subtypes
Target cell types: Goblet, Basal, Secretory, AT0

This script extracts specific subtypes from annotated epithelial cell data,
performs re-clustering and in-depth analysis with focus on heterogeneity
and functional states of each subtype

Author: Clinical-Bioinformatics Team
Date: 2025-11-12
Version: v1.1 (Added BBKNN integration)
"""

import sys
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import issparse, csr_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import time
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Try to load BBKNN
try:
    import bbknn
    HAS_BBKNN = True
    print("✓ BBKNN loaded successfully")
except Exception:
    HAS_BBKNN = False
    print("⚠️  BBKNN not available, will use standard neighbors")

# Set scanpy settings
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white')

print("✓ All libraries loaded successfully")

In [85]:
# ==================== CONFIGURATION ====================

# Input/Output
INPUT_H5AD_PATH = "/home/h2048/data/py/1029/bbknn_annotation/Epithelial/res_leiden_bbknn_res2.4/epithelial_annotated_resleiden_bbknn_res2.4.h5ad"
OUTPUT_DIR = "/home/h2048/data/py/1029/bbknn_celltype_analysis/Epithelial/subtype_analysis"
OVERWRITE_EXISTING = True

# Target cell types
TARGET_CELL_TYPES = ["goblet", "basal", "secretory"]
ANNOTATION_KEY = "cell_type"

# Batch and tissue keys
BATCH_KEY = "dataset"
TISSUE_KEY = "tissue"
TISSUE_SAMPLING_KEY = "tissue_sampling_method"
METADATA_COLS_TO_PLOT = [BATCH_KEY, TISSUE_KEY, TISSUE_SAMPLING_KEY]

# Preprocessing
MIN_CELLS = 10
MIN_GENES = 200
NORMALIZE_TOTAL = True
TARGET_SUM = 1e4
LOG_TRANSFORM = True
USE_HVG = True
N_TOP_GENES = 4000
HVG_FLAVOR = "seurat_v3"
SCALE_DATA = True
MAX_VALUE = 10

# Dimensionality reduction
N_PCS = 50
N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.5

# BBKNN configuration
USE_BBKNN = True
BBKNN_NEIGHBORS_WITHIN_BATCH = 3
BBKNN_TRIM = 25
BBKNN_METRIC = "cosine"

# Clustering
LEIDEN_RESOLUTIONS = [1.2, 1.6, 2.0, 2.4, 2.8]
DEFAULT_RESOLUTION = 2.4

# Marker analysis
RUN_FIND_MARKERS = True
MARKER_MIN_PCT = 0.25
MARKER_LOGFC_THRESHOLD = 0.25
TOP_N_MARKERS = 10

# Visualization
DPI = 300
FIGURE_FORMAT = "png"

print("✓ Configuration set")

✓ Configuration set


In [ ]:
# Subtype-specific marker genes
SUBTYPE_MARKERS = {
    'Basal': {
        'core': ['TP63', 'KRT5', 'KRT14', 'KRT15'],
        'proliferation': ['MKI67', 'TOP2A', 'PCNA'],
        'differentiation': ['KRT8', 'KRT18', 'KRT19'],
        'stem': ['ITGA6', 'NGFR', 'SOX2']
    },
    'Secretory': {
        'club': ['SCGB1A1', 'SCGB3A1', 'SCGB3A2'],
        'mucous': ['MUC5B', 'MUC5AC'],
        'serous': ['LTF', 'LYZ', 'DMBT1'],
        'antimicrobial': ['BPIFA1', 'BPIFB1', 'PIGR']
    },
    'Goblet': {
        'mucin': ['MUC5AC', 'MUC5B', 'MUC2'],
        'secretory': ['TFF1', 'TFF3', 'SPDEF'],
        'processing': ['AGR2', 'LYPD2'],
        'surface': ['ITLN1', 'FCGBP']
    },
    'AT0': {
        'progenitor': ['AGER', 'SFTPC', 'HOPX'],
        'at1_signature': ['AGER', 'PDPN', 'RTKN2'],
        'at2_signature': ['SFTPC', 'SFTPB', 'LAMP3'],
        'proliferation': ['MKI67', 'TOP2A']
    }
}

# Functional gene sets
FUNCTIONAL_GENE_SETS = {
    'Proliferation': ['MKI67', 'TOP2A', 'TK1', 'CENPW', 'PCNA'],
    'Stress_Response': ['HSPA1A', 'HSPA1B', 'HSPB1', 'HSP90AA1'],
    'Inflammatory': ['IL6', 'IL8', 'CXCL1', 'CXCL2', 'CXCL10'],
    'Ciliated_Transition': ['FOXJ1', 'RSPH1', 'DEUP1', 'FOXN4'],
    'EMT': ['VIM', 'CDH2', 'TWIST1', 'SNAI1', 'ZEB1'],
    'Metaplasia': ['KRT7', 'KRT13', 'KRT4']
}

print("✓ Marker genes and functional gene sets defined")

In [ ]:
# Create output directory
output_path = Path(OUTPUT_DIR)

if output_path.exists() and not OVERWRITE_EXISTING:
    raise FileExistsError(
        f"Output directory exists: {OUTPUT_DIR}\n"
        f"Set OVERWRITE_EXISTING=True to overwrite"
    )

output_path.mkdir(parents=True, exist_ok=True)
print(f"✓ Output directory ready: {OUTPUT_DIR}")

In [ ]:
print("\n" + "="*70)
print("Step 1: Loading Data and Extracting Target Cell Types")
print("="*70)

# Load data
print(f"\nReading file: {INPUT_H5AD_PATH}")
if not Path(INPUT_H5AD_PATH).exists():
    raise FileNotFoundError(f"File not found: {INPUT_H5AD_PATH}")

adata = sc.read_h5ad(INPUT_H5AD_PATH)
print(f"✓ Data loaded successfully")
print(f"   Total cells: {adata.n_obs:,}")
print(f"   Total genes: {adata.n_vars:,}")

# Check annotation key
if ANNOTATION_KEY not in adata.obs.columns:
    available_keys = [col for col in adata.obs.columns 
                     if 'anno' in col.lower() or 'type' in col.lower()]
    raise KeyError(
        f"Annotation column '{ANNOTATION_KEY}' not found\n"
        f"Available annotation-related columns: {available_keys}\n"
        f"Please modify ANNOTATION_KEY parameter"
    )

# Check available cell types
available_types = adata.obs[ANNOTATION_KEY].unique().tolist()
print(f"\nAvailable cell types: {available_types}")

# Try fuzzy matching if exact match fails
missing_types = [t for t in TARGET_CELL_TYPES if t not in available_types]
if missing_types:
    print(f"\n⚠️  Warning: Target types not found: {missing_types}")
    print("\nAttempting fuzzy matching...")
    matched_types = []
    for target in TARGET_CELL_TYPES:
        target_lower = target.lower()
        for avail in available_types:
            if target_lower in str(avail).lower() or str(avail).lower() in target_lower:
                matched_types.append(avail)
                print(f"  '{target}' matched to '{avail}'")
    if matched_types:
        TARGET_CELL_TYPES = list(dict.fromkeys(matched_types))
        print(f"\n✓ Using matched types: {TARGET_CELL_TYPES}")
    else:
        raise ValueError(
            f"Target cell types not found\n"
            f"Target: {TARGET_CELL_TYPES}\n"
            f"Available: {available_types}"
        )

# Extract subset
print(f"\nExtracting target cell types...")
mask = adata.obs[ANNOTATION_KEY].isin(TARGET_CELL_TYPES)
adata = adata[mask].copy()

print(f"✓ Subset extracted")
print(f"   Extracted cells: {adata.n_obs:,}")

# Cell type distribution
type_counts = adata.obs[ANNOTATION_KEY].value_counts()
print(f"\nCell type distribution:")
for cell_type, count in type_counts.items():
    pct = 100 * count / adata.n_obs
    print(f"   {cell_type}: {count:,} ({pct:.1f}%)")

# Check data structure
print("\nData structure:")
print(f"  .X: {adata.X.shape}, dtype={adata.X.dtype}")
if hasattr(adata, 'layers') and len(adata.layers) > 0:
    print(f"  .layers:")
    for key in adata.layers.keys():
        print(f"    - {key}: {adata.layers[key].shape}")
if adata.raw is not None:
    print(f"  .raw: {adata.raw.X.shape}")

In [ ]:
print("\n" + "="*70)
print("Step 1b: Plotting Initial UMAP (Before Re-analysis)")
print("="*70)

# Check if UMAP coordinates exist
if 'X_umap' in adata.obsm:
    figures_dir = output_path / "figures_initial"
    figures_dir.mkdir(exist_ok=True)
    print(f"\nInitial UMAP figures directory: {figures_dir}")
    
    # 1. Original cell types
    print("\n1. Original cell type distribution")
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(
        adata,
        color=ANNOTATION_KEY,
        ax=ax,
        show=False,
        legend_loc='right margin',
        title='Initial Cell Types (Before Re-analysis)'
    )
    plt.tight_layout()
    plt.savefig(figures_dir / f"initial_umap_cell_types.{FIGURE_FORMAT}", dpi=DPI)
    plt.close()
    
    # 2-4. Metadata columns
    for idx, col_name in enumerate(METADATA_COLS_TO_PLOT, start=2):
        if col_name not in adata.obs.columns:
            print(f"⚠️  Column '{col_name}' not found, skipping")
            continue
        
        print(f"{idx}. {col_name}")
        unique_vals = adata.obs[col_name].unique()
        n_unique = len(unique_vals)
        print(f"   Unique values: {n_unique}")
        
        if n_unique > 50:
            print(f"   ⚠️  Too many categories ({n_unique}), skipping visualization")
            continue
        
        fig, ax = plt.subplots(figsize=(10, 8))
        sc.pl.umap(
            adata,
            color=col_name,
            ax=ax,
            show=False,
            legend_loc='right margin',
            title=f'Initial {col_name} (Before Re-analysis)'
        )
        plt.tight_layout()
        safe_col_name = col_name.replace(' ', '_').replace('/', '_')
        plt.savefig(figures_dir / f"initial_umap_{safe_col_name}.{FIGURE_FORMAT}", dpi=DPI)
        plt.close()
    
    # 5. Combined plot
    print(f"{len(METADATA_COLS_TO_PLOT)+1}. Combined plot")
    available_cols = [ANNOTATION_KEY] + [col for col in METADATA_COLS_TO_PLOT 
                                         if col in adata.obs.columns]
    n_plots = len(available_cols)
    
    if n_plots > 0:
        ncols = 2
        nrows = (n_plots + 1) // 2
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*6, nrows*5))
        if n_plots == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        for idx, col_name in enumerate(available_cols):
            n_unique = adata.obs[col_name].nunique()
            if n_unique > 50:
                axes[idx].text(0.5, 0.5, f'{col_name}\nToo many categories ({n_unique})',
                             ha='center', va='center', transform=axes[idx].transAxes)
                axes[idx].axis('off')
                continue
            
            sc.pl.umap(
                adata,
                color=col_name,
                ax=axes[idx],
                show=False,
                title=col_name,
                legend_loc='right margin',
                frameon=False
            )
        
        for idx in range(n_plots, len(axes)):
            fig.delaxes(axes[idx])
        
        plt.tight_layout()
        plt.savefig(figures_dir / f"initial_umap_combined.{FIGURE_FORMAT}", 
                   dpi=DPI, bbox_inches='tight')
        plt.close()
    
    print(f"\n✓ Initial UMAP plots saved to: {figures_dir}")
    print(f"   Total figures generated: {len(list(figures_dir.glob('*.png')))}")
else:
    print("⚠️  No original UMAP coordinates found, skipping initial visualization")

In [ ]:
print("\n" + "="*70)
print("Step 2: Preprocessing Subset Data")
print("="*70)

# Save original counts to layers if not already present
if 'counts' not in adata.layers:
    if adata.raw is not None:
        print("Copying raw counts from .raw...")
        adata.layers['counts'] = adata.raw.X.copy()
    else:
        print("⚠️  Warning: No raw counts available, using current .X as counts")
        adata.layers['counts'] = adata.X.copy()

In [ ]:
# Gene filtering
print(f"\nGene filtering (min_cells={MIN_CELLS})...")
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
print(f"✓ Genes retained: {adata.n_vars:,}")

# Cell QC
print(f"\nCell QC (min_genes={MIN_GENES})...")
n_before = adata.n_obs
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
n_after = adata.n_obs
print(f"✓ Before filtering: {n_before:,} cells")
print(f"  After filtering: {n_after:,} cells")
print(f"  Removed: {n_before - n_after:,} cells")

In [88]:
# Cell 8.5: Remove Mitochondrial, Ribosomal and Hemoglobin Genes

print("\n" + "="*70)
print("Step 2b: Removing Mitochondrial, Ribosomal and Hemoglobin Genes")
print("="*70)

# Record genes before filtering
n_genes_before = adata.n_vars
print(f"\nGenes before MT/RP/HB filtering: {n_genes_before:,}")

# Identify mitochondrial genes (MT- for human, mt- for mouse)
mt_genes_mask = adata.var_names.str.startswith('MT-') | adata.var_names.str.startswith('mt-')
n_mt_genes = mt_genes_mask.sum()

if n_mt_genes > 0:
    mt_genes_list = adata.var_names[mt_genes_mask].tolist()
    print(f"\nMitochondrial genes found: {n_mt_genes}")
    print(f"  Examples: {', '.join(mt_genes_list[:10])}")
    if n_mt_genes > 10:
        print(f"  ... and {n_mt_genes - 10} more")
else:
    print(f"\nMitochondrial genes found: 0")

# Identify ribosomal genes (RPL and RPS)
rp_genes_mask = adata.var_names.str.startswith('RPL') | adata.var_names.str.startswith('RPS') | \
                adata.var_names.str.startswith('Rpl') | adata.var_names.str.startswith('Rps')
n_rp_genes = rp_genes_mask.sum()

if n_rp_genes > 0:
    rp_genes_list = adata.var_names[rp_genes_mask].tolist()
    print(f"\nRibosomal genes found: {n_rp_genes}")
    print(f"  Examples: {', '.join(rp_genes_list[:10])}")
    if n_rp_genes > 10:
        print(f"  ... and {n_rp_genes - 10} more")
else:
    print(f"\nRibosomal genes found: 0")

# Identify hemoglobin genes (HBA, HBB, HBD, HBE, HBG, HBM, HBQ, HBZ for human; Hba, Hbb for mouse)
hb_genes_mask = adata.var_names.str.match(r'^HB[ABDEGHQMZ]') | \
                adata.var_names.str.match(r'^Hb[ab]')
n_hb_genes = hb_genes_mask.sum()

if n_hb_genes > 0:
    hb_genes_list = adata.var_names[hb_genes_mask].tolist()
    print(f"\nHemoglobin genes found: {n_hb_genes}")
    print(f"  Genes: {', '.join(hb_genes_list)}")
else:
    print(f"\nHemoglobin genes found: 0")

# Combine masks to identify all genes to remove
genes_to_remove_mask = mt_genes_mask | rp_genes_mask | hb_genes_mask
n_genes_to_remove = genes_to_remove_mask.sum()

# Summary before removal
print(f"\n{'─'*70}")
print(f"Summary:")
print(f"  Mitochondrial genes: {n_mt_genes}")
print(f"  Ribosomal genes: {n_rp_genes}")
print(f"  Hemoglobin genes: {n_hb_genes}")
print(f"  Total to remove: {n_genes_to_remove}")
print(f"{'─'*70}")

# Remove MT, RP and HB genes
if n_genes_to_remove > 0:
    # Keep genes that are NOT MT, RP or HB
    genes_to_keep_mask = ~genes_to_remove_mask
    adata = adata[:, genes_to_keep_mask].copy()
    
    n_genes_after = adata.n_vars
    print(f"\nGenes after filtering: {n_genes_after:,}")
    print(f"Genes removed: {n_genes_before - n_genes_after:,} ({(n_genes_before - n_genes_after)/n_genes_before*100:.1f}%)")
    print("✓ MT, RP and HB genes removed successfully")
else:
    print("\nNo MT, RP or HB genes found, skipping removal")

print(f"\nFinal gene count: {adata.n_vars:,}")


Step 2b: Removing Mitochondrial, Ribosomal and Hemoglobin Genes

Genes before MT/RP/HB filtering: 36,302

Mitochondrial genes found: 0

Ribosomal genes found: 0

Hemoglobin genes found: 10
  Genes: HBQ1, HBEGF, HBZ, HBA2, HBG2, HBA1, HBM, HBE1, HBD, HBB

──────────────────────────────────────────────────────────────────────
Summary:
  Mitochondrial genes: 0
  Ribosomal genes: 0
  Hemoglobin genes: 10
  Total to remove: 10
──────────────────────────────────────────────────────────────────────

Genes after filtering: 36,292
Genes removed: 10 (0.0%)
✓ MT, RP and HB genes removed successfully

Final gene count: 36,292


In [71]:
# Normalization
if NORMALIZE_TOTAL:
    print(f"\nNormalization (target_sum={TARGET_SUM:.0e})...")
    adata.X = adata.layers['counts'].copy()
    sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
    
    if LOG_TRANSFORM:
        print("Log1p transformation...")
        sc.pp.log1p(adata)
    
    adata.layers['log1p'] = adata.X.copy()
    print("✓ Normalization complete")


Normalization (target_sum=1e+04)...
normalizing counts per cell
    finished (0:00:03)
Log1p transformation...
✓ Normalization complete


In [72]:
# Highly variable genes
if USE_HVG:
    print(f"\nHighly variable genes selection (n_top_genes={N_TOP_GENES})...")
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_TOP_GENES,
        flavor=HVG_FLAVOR,
        batch_key=BATCH_KEY if BATCH_KEY in adata.obs.columns else None
    )
    n_hvg = int(adata.var['highly_variable'].sum())
    print(f"✓ Highly variable genes: {n_hvg}")
    
    # Force retain key marker genes
    all_markers = []
    for markers_dict in SUBTYPE_MARKERS.values():
        for marker_list in markers_dict.values():
            all_markers.extend(marker_list)
    all_markers = list(set(all_markers))
    markers_in_data = [m for m in all_markers if m in adata.var_names]
    print(f"\nForcing retention of marker genes: {len(markers_in_data)}/{len(all_markers)}")
    
    if markers_in_data:
        adata.var.loc[markers_in_data, 'highly_variable'] = True
        n_hvg_final = int(adata.var['highly_variable'].sum())
        print(f"✓ Final highly variable genes: {n_hvg_final}")


Highly variable genes selection (n_top_genes=4000)...
If you pass `n_top_genes`, all cutoffs are ignored.
extracting highly variable genes
--> added
    'highly_variable', boolean vector (adata.var)
    'highly_variable_rank', float vector (adata.var)
    'means', float vector (adata.var)
    'variances', float vector (adata.var)
    'variances_norm', float vector (adata.var)
✓ Highly variable genes: 4000

Forcing retention of marker genes: 39/39
✓ Final highly variable genes: 4011


In [73]:
# Scaling
if SCALE_DATA:
    print(f"\nScaling (max_value={MAX_VALUE})...")
    sc.pp.scale(adata, max_value=MAX_VALUE,zero_center=True)
    print("✓ Scaling complete")

print(f"\nPreprocessing complete")
print(f"   Final cells: {adata.n_obs:,}")
print(f"   Final genes: {adata.n_vars:,}")


Scaling (max_value=10)...
... as `zero_center=True`, sparse input is densified and may lead to large memory consumption
✓ Scaling complete

Preprocessing complete
   Final cells: 147,596
   Final genes: 36,302


In [74]:
print("\n" + "="*70)
print("Step 3: Dimensionality Reduction")
print("="*70)

# PCA
print(f"\nPCA analysis (n_pcs={N_PCS})...")
if USE_HVG:
    sc.tl.pca(adata, n_comps=N_PCS, use_highly_variable=True)
else:
    sc.tl.pca(adata, n_comps=N_PCS)
print("✓ PCA complete")


Step 3: Dimensionality Reduction

PCA analysis (n_pcs=50)...
computing PCA
    on highly variable genes
    with n_comps=50
    finished (0:01:35)
✓ PCA complete


In [89]:
# Build neighbor graph (BBKNN preferred)
use_bbknn_now = (
    USE_BBKNN and
    HAS_BBKNN and
    (BATCH_KEY in adata.obs.columns) and
    (adata.obs[BATCH_KEY].nunique() > 1)
)

# Ensure batch column is categorical
if BATCH_KEY in adata.obs.columns:
    if not pd.api.types.is_categorical_dtype(adata.obs[BATCH_KEY]):
        adata.obs[BATCH_KEY] = adata.obs[BATCH_KEY].astype('category')

if use_bbknn_now:
    print(f"\nUsing BBKNN to build neighbor graph (batch_key='{BATCH_KEY}', "
          f"neighbors_within_batch={BBKNN_NEIGHBORS_WITHIN_BATCH}, n_pcs={N_PCS})...")
    try:
        bbknn.bbknn(
            adata,
            batch_key=BATCH_KEY,
            neighbors_within_batch=BBKNN_NEIGHBORS_WITHIN_BATCH,
            n_pcs=N_PCS,
            metric=BBKNN_METRIC,
            trim=BBKNN_TRIM
        )
        adata.uns['integration_method'] = 'bbknn'
        print("✓ BBKNN neighbor graph constructed")
    except Exception as e:
        print(f"⚠️  BBKNN failed, using standard neighbors. Reason: {e}")
        sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS)
        adata.uns['integration_method'] = 'neighbors'
        print("✓ Standard neighbor graph constructed")
else:
    if not HAS_BBKNN and USE_BBKNN:
        print("⚠️  bbknn not installed, using standard neighbors")
    elif BATCH_KEY not in adata.obs.columns or adata.obs[BATCH_KEY].nunique() <= 1:
        print("⚠️  Missing valid batch column or batch count ≤1, using standard neighbors")
    
    print(f"\nBuilding standard neighbor graph (n_neighbors={N_NEIGHBORS})...")
    sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS)
    adata.uns['integration_method'] = 'neighbors'
    print("✓ Standard neighbor graph constructed")


Using BBKNN to build neighbor graph (batch_key='dataset', neighbors_within_batch=3, n_pcs=50)...
computing batch balanced neighbors
	finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:02:42)
✓ BBKNN neighbor graph constructed


In [90]:
# UMAP
print(f"\nUMAP dimensionality reduction (min_dist={UMAP_MIN_DIST})...")
sc.tl.umap(adata, min_dist=UMAP_MIN_DIST)
print("✓ UMAP complete")


UMAP dimensionality reduction (min_dist=0.5)...
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm) (0:02:30)
✓ UMAP complete


In [91]:
print("\n" + "="*70)
print("Step 4: Multi-Resolution Clustering")
print("="*70)

print(f"\nLeiden clustering (resolutions: {LEIDEN_RESOLUTIONS})...")
for res in tqdm(LEIDEN_RESOLUTIONS, desc="Clustering"):
    key = f'leiden_r{res}'
    sc.tl.leiden(adata, resolution=res, key_added=key)
    n_clusters = adata.obs[key].nunique()
    print(f"  Resolution {res}: {n_clusters} clusters")

# Set default clustering result
default_key = f'leiden_r{DEFAULT_RESOLUTION}'
adata.obs['leiden'] = adata.obs[default_key]

print(f"\n✓ Clustering complete")
print(f"   Default resolution: {DEFAULT_RESOLUTION}")
print(f"   Default cluster count: {adata.obs['leiden'].nunique()}")


Step 4: Multi-Resolution Clustering

Leiden clustering (resolutions: [1.2, 1.6, 2.0, 2.4, 2.8])...


Clustering:   0%|          | 0/5 [00:00<?, ?it/s]

running Leiden clustering
    finished: found 29 clusters and added
    'leiden_r1.2', the cluster labels (adata.obs, categorical) (0:01:27)


Clustering:  20%|██        | 1/5 [01:27<05:50, 87.58s/it]

  Resolution 1.2: 29 clusters
running Leiden clustering
    finished: found 34 clusters and added
    'leiden_r1.6', the cluster labels (adata.obs, categorical) (0:01:21)


...

  Resolution 2.0: 40 clusters
running Leiden clustering
    finished: found 47 clusters and added
    'leiden_r2.4', the cluster labels (adata.obs, categorical) (0:02:09)


Clustering:  80%|████████  | 4/5 [06:50<01:49, 109.60s/it]

  Resolution 2.4: 47 clusters
running Leiden clustering
    finished: found 52 clusters and added
    'leiden_r2.8', the cluster labels (adata.obs, categorical) (0:01:11)


Clustering: 100%|██████████| 5/5 [08:01<00:00, 96.35s/it] 

  Resolution 2.8: 52 clusters

✓ Clustering complete
   Default resolution: 2.4
   Default cluster count: 47


In [92]:
print("\n" + "="*70)
print("Step 6: Finding Cluster Markers")
print("="*70)

if RUN_FIND_MARKERS:
    # Ensure using log1p data for differential analysis
    if 'log1p' in adata.layers:
        print("Using log1p layer for differential analysis...")
        adata.X = adata.layers['log1p'].copy()
    
    print(f"\nFinding differential genes (method=wilcoxon)...")
    print(f"   min_pct: {MARKER_MIN_PCT}")
    print(f"   logfc_threshold: {MARKER_LOGFC_THRESHOLD}")
    
    try:
        sc.tl.rank_genes_groups(
            adata,
            groupby='leiden',
            method='wilcoxon',
            key_added='rank_genes_leiden',
            pts=True
        )
        print("✓ Differential gene analysis complete")
        
        # Save results
        marker_file = output_path / "cluster_markers.xlsx"
        print(f"\nSaving marker genes to: {marker_file}")
        
        result = adata.uns['rank_genes_leiden']
        groups = result['names'].dtype.names
        
        try:
            with pd.ExcelWriter(marker_file, engine='openpyxl') as writer:
                for group in groups:
                    df = pd.DataFrame({
                        'gene': result['names'][group][:50],
                        'scores': result['scores'][group][:50],
                        'logfoldchanges': result['logfoldchanges'][group][:50],
                        'pvals': result['pvals'][group][:50],
                        'pvals_adj': result['pvals_adj'][group][:50],
                        'pct_in_group': result['pts'][group][:50]
                    })
                    df.to_excel(writer, sheet_name=f'Cluster_{group}', index=False)
            print("✓ Marker genes saved (xlsx)")
        except Exception as e:
            # Fallback to CSV if openpyxl not available
            csv_dir = output_path / "markers_csv"
            csv_dir.mkdir(exist_ok=True)
            for group in groups:
                df = pd.DataFrame({
                    'gene': result['names'][group][:50],
                    'scores': result['scores'][group][:50],
                    'logfoldchanges': result['logfoldchanges'][group][:50],
                    'pvals': result['pvals'][group][:50],
                    'pvals_adj': result['pvals_adj'][group][:50],
                    'pct_in_group': result['pts'][group][:50]
                })
                df.to_csv(csv_dir / f"Cluster_{group}.csv", index=False)
            print(f"⚠️  Failed to save xlsx ({e}), saved as CSV to: {csv_dir}")
        
    except Exception as e:
        print(f"⚠️  Differential analysis failed: {e}")
else:
    print("\nSkipping marker gene analysis (RUN_FIND_MARKERS=False)")


Step 6: Finding Cluster Markers
Using log1p layer for differential analysis...



Finding differential genes (method=wilcoxon)...
   min_pct: 0.25
   logfc_threshold: 0.25
ranking genes
    finished: added to `.uns['rank_genes_leiden']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:18:53)
✓ Differential gene analysis complete

Saving marker genes to: /home/h2048/data/py/1029/bbknn_celltype_analysis/Epithelial/subtype_analysis/cluster_markers.xlsx
✓ Marker genes saved (xlsx)


In [93]:
print("\n" + "="*70)
print("Step 7: Generating Visualizations (After Re-analysis)")
print("="*70)

figures_dir = output_path / "figures"
figures_dir.mkdir(exist_ok=True)
print(f"\nFigures directory: {figures_dir}")


Step 7: Generating Visualizations (After Re-analysis)

Figures directory: /home/h2048/data/py/1029/bbknn_celltype_analysis/Epithelial/subtype_analysis/figures


In [94]:
# 1. UMAP - Cell types
print("\n1. UMAP - Original cell types")
fig, ax = plt.subplots(figsize=(10, 8))
sc.pl.umap(
    adata,
    color=ANNOTATION_KEY,
    ax=ax,
    show=False,
    legend_loc='right margin',
    title='Cell Types (After Re-analysis)'
)
plt.tight_layout()
plt.savefig(figures_dir / f"umap_cell_types.{FIGURE_FORMAT}", dpi=DPI)
plt.close()
print("✓ Saved")

# 2. UMAP - Leiden clustering
print("2. UMAP - Leiden clustering")
fig, ax = plt.subplots(figsize=(10, 8))
sc.pl.umap(
    adata,
    color='leiden',
    ax=ax,
    show=False,
    legend_loc='right margin',
    title='Leiden Clusters (After Re-analysis)'
)
plt.tight_layout()
plt.savefig(figures_dir / f"umap_leiden.{FIGURE_FORMAT}", dpi=DPI)
plt.close()
print("✓ Saved")


1. UMAP - Original cell types
✓ Saved
2. UMAP - Leiden clustering
✓ Saved


In [95]:
# 3-5. UMAP - Metadata columns
plot_counter = 3
for col_name in METADATA_COLS_TO_PLOT:
    if col_name not in adata.obs.columns:
        print(f"⚠️  Column '{col_name}' not found, skipping")
        continue
    
    print(f"{plot_counter}. UMAP - {col_name}")
    unique_vals = adata.obs[col_name].unique()
    n_unique = len(unique_vals)
    print(f"   Unique values: {n_unique}")
    
    if n_unique > 50:
        print(f"   ⚠️  Too many categories ({n_unique}), skipping visualization")
        plot_counter += 1
        continue
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(
        adata,
        color=col_name,
        ax=ax,
        show=False,
        legend_loc='right margin',
        title=f'{col_name} (After Re-analysis)'
    )
    plt.tight_layout()
    safe_col_name = col_name.replace(' ', '_').replace('/', '_')
    plt.savefig(figures_dir / f"umap_{safe_col_name}.{FIGURE_FORMAT}", dpi=DPI)
    plt.close()
    print("✓ Saved")
    plot_counter += 1

3. UMAP - dataset
   Unique values: 19
✓ Saved
4. UMAP - tissue
   Unique values: 4
✓ Saved
5. UMAP - tissue_sampling_method
   Unique values: 4
✓ Saved


In [96]:
# Violin plot - Key marker genes
print(f"\n{plot_counter}. Violin plot - Key marker genes")

# Collect all markers
all_markers = []
for cell_type, markers_dict in SUBTYPE_MARKERS.items():
    for category, genes in markers_dict.items():
        all_markers.extend(genes)
all_markers = list(set(all_markers))
markers_in_data = [m for m in all_markers if m in adata.var_names]

if markers_in_data:
    markers_to_plot = markers_in_data[:min(20, len(markers_in_data))]
    print(f"   Plotting {len(markers_to_plot)} markers")
    
    fig = plt.figure(figsize=(4*len(markers_to_plot), 6))
    sc.pl.violin(
        adata,
        keys=markers_to_plot,
        groupby='leiden',
        rotation=90,
        show=False
    )
    plt.tight_layout()
    plt.savefig(figures_dir / f"violin_markers.{FIGURE_FORMAT}", dpi=DPI, bbox_inches='tight')
    plt.close()
    print("✓ Saved")
else:
    print("   No markers found in data")

plot_counter += 1


6. Violin plot - Key marker genes
   Plotting 20 markers
✓ Saved


<Figure size 6400x480 with 0 Axes>

In [97]:
# Dotplot - Marker genes
print(f"\n{plot_counter}. Dotplot - Marker gene expression")

if markers_in_data:
    print(f"   Plotting {len(markers_in_data)} markers")
    
    fig = plt.figure(figsize=(max(12, len(markers_in_data)*0.3), 8))
    sc.pl.dotplot(
        adata,
        var_names=markers_in_data,
        groupby='leiden',
        show=False,
        standard_scale='var'
    )
    plt.tight_layout()
    plt.savefig(figures_dir / f"dotplot_markers.{FIGURE_FORMAT}", dpi=DPI, bbox_inches='tight')
    plt.close()
    print("✓ Saved")
else:
    print("   No markers found in data")

plot_counter += 1


7. Dotplot - Marker gene expression
   Plotting 39 markers
✓ Saved


<Figure size 960x640 with 0 Axes>

In [84]:
# Cell type composition heatmap
print(f"\n{plot_counter}. Cell type composition heatmap")

ct_cluster = pd.crosstab(
    adata.obs[ANNOTATION_KEY],
    adata.obs['leiden'],
    normalize='columns'
) * 100

fig, ax = plt.subplots(figsize=(max(10, ct_cluster.shape[1]*0.6), 6))
sns.heatmap(
    ct_cluster,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    ax=ax,
    cbar_kws={'label': 'Percentage (%)'}
)
ax.set_xlabel('Leiden Cluster')
ax.set_ylabel('Cell Type')
ax.set_title('Cell Type Composition by Cluster')
plt.tight_layout()
plt.savefig(figures_dir / f"heatmap_composition.{FIGURE_FORMAT}", dpi=DPI)
plt.close()
print("✓ Saved")

plot_counter += 1


8. Cell type composition heatmap
✓ Saved


In [ ]:
# Cell count statistics bar plots
print(f"\n{plot_counter}. Cell count statistics")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cell type counts
type_counts = adata.obs[ANNOTATION_KEY].value_counts()
type_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Cell Type')
axes[0].set_ylabel('Cell Count')
axes[0].set_title('Cell Count by Type')
axes[0].tick_params(axis='x', rotation=45)

# Cluster counts
cluster_counts = adata.obs['leiden'].value_counts().sort_index()
cluster_counts.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_xlabel('Leiden Cluster')
axes[1].set_ylabel('Cell Count')
axes[1].set_title('Cell Count by Cluster')

plt.tight_layout()
plt.savefig(figures_dir / f"barplot_cell_counts.{FIGURE_FORMAT}", dpi=DPI)
plt.close()
print("✓ Saved")

plot_counter += 1

In [ ]:
# Multi-resolution clustering comparison
print(f"\n{plot_counter}. Multi-resolution clustering comparison")

resolution_keys = [f'leiden_r{res}' for res in LEIDEN_RESOLUTIONS]
existing_keys = [k for k in resolution_keys if k in adata.obs.columns]

if len(existing_keys) > 1:
    n_res = len(existing_keys)
    ncols = min(3, n_res)
    nrows = (n_res + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*4))
    axes = axes.flatten() if n_res > 1 else [axes]
    
    for idx, res_key in enumerate(existing_keys):
        res_value = res_key.replace('leiden_r', '')
        sc.pl.umap(
            adata,
            color=res_key,
            ax=axes[idx],
            show=False,
            title=f'Resolution {res_value}',
            legend_loc='right margin'
        )
    
    for idx in range(n_res, len(axes)):
        fig.delaxes(axes[idx])
    
    plt.tight_layout()
    plt.savefig(figures_dir / f"umap_multi_resolution.{FIGURE_FORMAT}", dpi=DPI)
    plt.close()
    print("✓ Saved")
else:
    print("   Insufficient resolution keys for comparison")

plot_counter += 1

In [ ]:
# Combined plot after analysis
print(f"\n{plot_counter}. Combined UMAP after analysis")

cols_to_show = [ANNOTATION_KEY, 'leiden']
for col in METADATA_COLS_TO_PLOT:
    if col in adata.obs.columns and adata.obs[col].nunique() <= 50:
        cols_to_show.append(col)

n_plots = len(cols_to_show)
if n_plots > 0:
    ncols = 3
    nrows = (n_plots + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*4))
    if n_plots == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for idx, col_name in enumerate(cols_to_show):
        sc.pl.umap(
            adata,
            color=col_name,
            ax=axes[idx],
            show=False,
            title=col_name,
            legend_loc='right margin',
            frameon=False
        )
    
    for idx in range(n_plots, len(axes)):
        fig.delaxes(axes[idx])
    
    plt.tight_layout()
    plt.savefig(figures_dir / f"umap_combined_after_analysis.{FIGURE_FORMAT}", 
               dpi=DPI, bbox_inches='tight')
    plt.close()
    print("✓ Saved")

print(f"\n✓ All figures saved to: {figures_dir}")
print(f"   Total figures generated: {len(list(figures_dir.glob('*.png')))}")

In [ ]:
print("\n" + "="*70)
print("Step 8: Generating Summary Report")
print("="*70)

report_file = output_path / "analysis_summary.txt"

with open(report_file, 'w', encoding='utf-8') as f:
    f.write("="*70 + "\n")
    f.write("Epithelial Cell Subtype Analysis - Summary Report\n")
    f.write("="*70 + "\n\n")
    
    # Basic information
    f.write("1. Basic Information\n")
    f.write("-"*70 + "\n")
    f.write(f"Analysis date: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Target cell types: {', '.join(TARGET_CELL_TYPES)}\n")
    f.write(f"Total cells: {adata.n_obs:,}\n")
    f.write(f"Total genes: {adata.n_vars:,}\n\n")
    
    # Cell type distribution
    f.write("2. Cell Type Distribution\n")
    f.write("-"*70 + "\n")
    type_counts = adata.obs[ANNOTATION_KEY].value_counts()
    for cell_type, count in type_counts.items():
        pct = 100 * count / adata.n_obs
        f.write(f"   {cell_type}: {count:,} ({pct:.2f}%)\n")
    f.write("\n")
    
    # Leiden clustering results
    f.write("3. Leiden Clustering Results\n")
    f.write("-"*70 + "\n")
    f.write(f"Default resolution: {DEFAULT_RESOLUTION}\n")
    f.write(f"Cluster count: {adata.obs['leiden'].nunique()}\n\n")
    cluster_counts = adata.obs['leiden'].value_counts().sort_index()
    f.write("Cell count per cluster:\n")
    for cluster, count in cluster_counts.items():
        pct = 100 * count / adata.n_obs
        f.write(f"   Cluster {cluster}: {count:,} ({pct:.2f}%)\n")
    f.write("\n")
    
    # Cluster cell type composition
    f.write("4. Cluster Cell Type Composition\n")
    f.write("-"*70 + "\n")
    ct_cluster = pd.crosstab(
        adata.obs[ANNOTATION_KEY],
        adata.obs['leiden']
    )
    for cluster in sorted(adata.obs['leiden'].unique()):
        f.write(f"\nCluster {cluster}:\n")
        cluster_data = ct_cluster[cluster]
        cluster_total = cluster_data.sum()
        for cell_type, count in cluster_data.items():
            if count > 0:
                pct = 100 * count / cluster_total
                f.write(f"   {cell_type}: {count:,} ({pct:.1f}%)\n")
    f.write("\n")
    
    # Batch distribution
    if BATCH_KEY in adata.obs.columns:
        f.write("5. Batch Distribution\n")
        f.write("-"*70 + "\n")
        batch_counts = adata.obs[BATCH_KEY].value_counts()
        for batch, count in batch_counts.items():
            pct = 100 * count / adata.n_obs
            f.write(f"   {batch}: {count:,} ({pct:.2f}%)\n")
        f.write("\n")
    
    # Tissue distribution
    if TISSUE_KEY in adata.obs.columns:
        f.write("6. Tissue Distribution\n")
        f.write("-"*70 + "\n")
        tissue_counts = adata.obs[TISSUE_KEY].value_counts()
        for tissue, count in tissue_counts.items():
            pct = 100 * count / adata.n_obs
            f.write(f"   {tissue}: {count:,} ({pct:.2f}%)\n")
        f.write("\n")
    
    # Analysis parameters
    f.write("7. Analysis Parameters\n")
    f.write("-"*70 + "\n")
    f.write(f"Highly variable genes: {N_TOP_GENES}\n")
    f.write(f"PCA components: {N_PCS}\n")
    f.write(f"Integration method: {adata.uns.get('integration_method','unknown')}\n")
    if adata.uns.get('integration_method','') == 'bbknn':
        f.write(f"BBKNN neighbors_within_batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}\n")
        f.write(f"BBKNN metric: {BBKNN_METRIC}\n")
        f.write(f"BBKNN trim: {BBKNN_TRIM}\n")
    else:
        f.write(f"Standard neighbors n_neighbors: {N_NEIGHBORS}\n")
    f.write(f"Clustering resolutions: {LEIDEN_RESOLUTIONS}\n")
    f.write(f"Default resolution: {DEFAULT_RESOLUTION}\n")
    f.write("\n")
    
    f.write("="*70 + "\n")
    f.write("Report End\n")
    f.write("="*70 + "\n")

print(f"✓ Summary report saved: {report_file}")

In [103]:
print("\n" + "="*70)
print("Step 9: Saving Results")
print("="*70)

# 1. Save complete AnnData object
h5ad_file = output_path / "adata_subtype_analyzed.h5ad"
print(f"\nSaving AnnData object: {h5ad_file}")
adata.write_h5ad(h5ad_file, compression='gzip', compression_opts=1)
print("✓ AnnData saved")

# 2. Save metadata
metadata_file = output_path / "metadata.csv"
print(f"\nSaving metadata: {metadata_file}")
adata.obs.to_csv(metadata_file)
print("✓ Metadata saved")

# 3. Save clustering results
clustering_file = output_path / "clustering_results.csv"
print(f"\nSaving clustering results: {clustering_file}")
cluster_cols = [ANNOTATION_KEY] + [col for col in adata.obs.columns if 'leiden' in col.lower()]
if BATCH_KEY in adata.obs.columns:
    cluster_cols.append(BATCH_KEY)
if TISSUE_KEY in adata.obs.columns:
    cluster_cols.append(TISSUE_KEY)
cluster_df = adata.obs[cluster_cols].copy()
cluster_df.to_csv(clustering_file)
print("✓ Clustering results saved")

print(f"\n✓ All results saved to: {OUTPUT_DIR}")


Step 9: Saving Results

Saving AnnData object: /home/h2048/data/py/1029/bbknn_celltype_analysis/Epithelial/subtype_analysis/adata_subtype_analyzed.h5ad
✓ AnnData saved

Saving metadata: /home/h2048/data/py/1029/bbknn_celltype_analysis/Epithelial/subtype_analysis/metadata.csv
✓ Metadata saved

Saving clustering results: /home/h2048/data/py/1029/bbknn_celltype_analysis/Epithelial/subtype_analysis/clustering_results.csv
✓ Clustering results saved

✓ All results saved to: /home/h2048/data/py/1029/bbknn_celltype_analysis/Epithelial/subtype_analysis


In [98]:
# ==================== Annotation Helper Functions ====================

def standardize_cluster_id(cluster_value):
    """
    Standardize cluster ID format for consistent mapping
    Converts various types (int, float, str) to consistent string format
    """
    s = str(cluster_value).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except (ValueError, AttributeError):
        pass
    return s


def generate_color_palette(n_colors, base_palette=None):
    """
    Generate color palette for n_colors, extending beyond predefined colors if needed
    """
    if base_palette is None:
        base_palette = {
            "Basal": "#E41A1C",
            "Secretory": "#377EB8",
            "Ciliated": "#4DAF4A",
            "Goblet": "#984EA3",
            "Club": "#FF7F00",
            "Ionocyte": "#FFFF33",
            "Serous": "#A65628",
            "PNEC": "#F781BF",
            "AT1": "#66C2A5",
            "AT2": "#FC8D62",
            "Transitional": "#8DA0CB",
            "Proliferating": "#E78AC3",
            "Unknown": "#CCCCCC"
        }
    
    colors = []
    base_colors = list(base_palette.values())
    
    if n_colors <= len(base_colors):
        return base_colors[:n_colors]
    
    colors.extend(base_colors)
    remaining = n_colors - len(colors)
    
    if remaining > 0:
        cmaps = [plt.cm.tab20, plt.cm.tab20b, plt.cm.tab20c, plt.cm.Set3]
        cmap_colors = []
        for cmap in cmaps:
            n_cmap = cmap.N if hasattr(cmap, 'N') else 20
            for i in range(n_cmap):
                cmap_colors.append(plt.matplotlib.colors.to_hex(cmap(i)))
        colors.extend(cmap_colors[:remaining])
    
    return colors[:n_colors]


print("✓ Annotation helper functions defined")

✓ Annotation helper functions defined


In [99]:
# Define epithelial cell subtype markers
EPITHELIAL_MARKERS = [
    # Basal cells
    'TP63', 'KRT5', 'KRT14', 'KRT15',
    
    # Secretory cells (Club/Secretory)
    'SCGB1A1', 'SERPINB3', 'SCGB3A2', 'SCGB3A1', 'TCN1', 'ASRGL1', 'BPIFB1',
    
    # Ciliated cells
    'FOXJ1', 'RSPH1', 'PIFO', 'BEST4', 'C20orf85', 'C9orf24', 'CYP2F1',
    
    # Goblet cells
    'MUC5AC', 'SPDEF', 'LYPD2', 'ITLN1', 'TFF3',
    
    # Neuroendocrine (PNEC)
    'ASCL1', 'GRP', 'CHGA', 'CHGB',
    
    # Tuft cells
    'POU2F3', 'ASCL2', 'TRPM5',
    
    # Ionocytes
    'CFTR', 'FOXI1', 'ASCL3', 'BSND', 'IGF1', 'CLCNKB', 'PDE1C',
    
    # AT1 (if lung tissue)
    'AGER', 'RTKN2', 'CLIC5', 'SPOCK2', 'TIMP3', 'PDPN',
    
    # AT2 (if lung tissue)
    'SFTPC', 'LAMP3', 'SFTPB', 'SFTA2', 'NAPSA',
    
    # Transitional/Progenitor
    'VIM', 'SOX9', 'KRT8', 'KRT18', 'KRT19',
    
    # Smooth muscle/Myoepithelial
    'KRT14', 'MYH11', 'ACTA2', 'MYLK',
    
    # Serous cells
    'DMBT1', 'RNASE1', 'LYZ', 'LTF', 'PIP', 'CCL28',
    
    # Additional mucous
    'MUC5B',
    
    # Proliferation
    'MKI67', 'TOP2A', 'PCNA'
]

print(f"✓ Epithelial markers defined: {len(EPITHELIAL_MARKERS)} genes")

✓ Epithelial markers defined: 67 genes


In [100]:
print("\n" + "="*70)
print("Annotation Step 1: Checking Marker Gene Availability")
print("="*70)

# Build case-insensitive gene mapping
upper_to_original = {}
for gene in adata.var_names:
    gene_upper = str(gene).upper()
    if gene_upper not in upper_to_original:
        upper_to_original[gene_upper] = gene

# Check which markers are available
available_markers = []
missing_markers = []

for marker in EPITHELIAL_MARKERS:
    marker_upper = marker.upper()
    if marker_upper in upper_to_original:
        available_markers.append(upper_to_original[marker_upper])
    else:
        missing_markers.append(marker)

# Report
print(f"\nTotal markers queried: {len(EPITHELIAL_MARKERS)}")
print(f"Available in dataset: {len(available_markers)} ({len(available_markers)/len(EPITHELIAL_MARKERS)*100:.1f}%)")
print(f"Missing: {len(missing_markers)} ({len(missing_markers)/len(EPITHELIAL_MARKERS)*100:.1f}%)")

if missing_markers:
    print(f"\nMissing markers ({len(missing_markers)}):")
    for gene in missing_markers[:15]:
        print(f"  - {gene}")
    if len(missing_markers) > 15:
        print(f"  ... and {len(missing_markers)-15} more")

if len(available_markers) < len(EPITHELIAL_MARKERS) * 0.5:
    print("\n⚠️  Warning: Less than 50% of markers available!")
    print("Consider checking gene naming conventions or updating marker list")


Annotation Step 1: Checking Marker Gene Availability

Total markers queried: 67
Available in dataset: 67 (100.0%)
Missing: 0 (0.0%)


In [101]:
print("\n" + "="*70)
print("Annotation Step 2: Generating Marker Gene DotPlot")
print("="*70)

if len(available_markers) == 0:
    print("⚠️  No markers available for dotplot")
else:
    # Create figures directory
    figures_dir = output_path / "figures_annotation"
    figures_dir.mkdir(exist_ok=True)
    
    print(f"\nCreating dotplot with {len(available_markers)} markers...")
    print(f"Using cluster key: leiden")
    
    # Calculate figure size based on number of markers and clusters
    n_clusters = adata.obs['leiden'].nunique()
    fig_width = max(12, len(available_markers) * 0.35)
    fig_height = max(8, n_clusters * 0.5)
    
    # Create dotplot
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    sc.pl.dotplot(
        adata,
        var_names=available_markers,
        groupby='leiden',
        ax=ax,
        show=False,
        standard_scale='var',
        cmap='Reds'
    )
    ax.set_title(f'Marker Gene Expression - Epithelial Subtypes\n'
                f'n={adata.n_obs:,} cells', fontsize=14)
    
    plt.tight_layout()
    output_file = figures_dir / f"dotplot_epithelial_markers.{FIGURE_FORMAT}"
    plt.savefig(output_file, dpi=DPI, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Dotplot saved to: {output_file}")


Annotation Step 2: Generating Marker Gene DotPlot

Creating dotplot with 67 markers...
Using cluster key: leiden
✓ Dotplot saved to: /home/h2048/data/py/1029/bbknn_celltype_analysis/Epithelial/subtype_analysis/figures_annotation/dotplot_epithelial_markers.png


In [102]:
print("\n" + "="*70)
print("Annotation Step 2b: Generating Marker Gene Feature Plots (UMAP)")
print("="*70)

if len(available_markers) == 0:
    print("⚠️  No markers available for feature plots")
else:
    figures_dir = output_path / "figures_annotation"
    figures_dir.mkdir(exist_ok=True)
    
    print(f"\nGenerating feature plots for {len(available_markers)} markers...")
    
    # Option 1: Plot all markers in batches
    MARKERS_PER_PLOT = 16  # 4x4 grid
    n_batches = int(np.ceil(len(available_markers) / MARKERS_PER_PLOT))
    
    print(f"Creating {n_batches} feature plot batch(es)...")
    
    for batch_idx in range(n_batches):
        start_idx = batch_idx * MARKERS_PER_PLOT
        end_idx = min((batch_idx + 1) * MARKERS_PER_PLOT, len(available_markers))
        batch_markers = available_markers[start_idx:end_idx]
        
        if len(batch_markers) == 0:
            continue
        
        print(f"\n  Batch {batch_idx + 1}/{n_batches}: {len(batch_markers)} markers")
        
        # Calculate grid layout
        n_markers = len(batch_markers)
        ncols = 4
        nrows = int(np.ceil(n_markers / ncols))
        
        # Create figure
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*4))
        if n_markers == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        # Plot each marker
        for idx, marker in enumerate(batch_markers):
            sc.pl.umap(
                adata,
                color=marker,
                ax=axes[idx],
                show=False,
                title=marker,
                frameon=False,
                cmap='viridis',
                vmin=0,
                vmax='p99'  # Use 99th percentile to avoid outliers
            )
        
        # Remove empty subplots
        for idx in range(n_markers, len(axes)):
            fig.delaxes(axes[idx])
        
        plt.tight_layout()
        output_file = figures_dir / f"featureplot_markers_batch{batch_idx + 1}.{FIGURE_FORMAT}"
        plt.savefig(output_file, dpi=DPI, bbox_inches='tight')
        plt.close()
        print(f"    ✓ Saved: {output_file.name}")
    
    # Option 2: Create separate plots by marker category (if using subtype markers)
    print(f"\n  Generating category-specific feature plots...")
    
    # Define key marker categories for epithelial cells
    marker_categories = {
        'Basal': ['TP63', 'KRT5', 'KRT14', 'KRT15'],
        'Secretory': ['SCGB1A1', 'SCGB3A2', 'MUC5B'],
        'Ciliated': ['FOXJ1', 'RSPH1', 'PIFO'],
        'Goblet': ['MUC5AC', 'SPDEF', 'TFF3'],
        'AT1': ['AGER', 'PDPN', 'RTKN2'],
        'AT2': ['SFTPC', 'SFTPB', 'NAPSA'],
        'Proliferating': ['MKI67', 'TOP2A', 'PCNA']
    }
    
    for category, markers in marker_categories.items():
        # Check which markers are available
        markers_available = [m for m in markers if m in available_markers]
        
        if len(markers_available) == 0:
            print(f"    {category}: No markers available, skipping")
            continue
        
        print(f"    {category}: {len(markers_available)} markers")
        
        # Create plot
        n_markers = len(markers_available)
        ncols = min(3, n_markers)
        nrows = int(np.ceil(n_markers / ncols))
        
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*4.5))
        if n_markers == 1:
            axes = [axes]
        elif nrows == 1:
            axes = axes
        else:
            axes = axes.flatten()
        
        for idx, marker in enumerate(markers_available):
            sc.pl.umap(
                adata,
                color=marker,
                ax=axes[idx],
                show=False,
                title=f'{marker} ({category})',
                frameon=False,
                cmap='viridis',
                vmin=0,
                vmax='p99'
            )
        
        # Remove empty subplots
        for idx in range(n_markers, len(axes) if isinstance(axes, np.ndarray) else 1):
            if isinstance(axes, np.ndarray):
                fig.delaxes(axes[idx])
        
        plt.suptitle(f'{category} Marker Genes', fontsize=16, y=1.02)
        plt.tight_layout()
        
        safe_category = category.replace(' ', '_').replace('/', '_')
        output_file = figures_dir / f"featureplot_{safe_category}.{FIGURE_FORMAT}"
        plt.savefig(output_file, dpi=DPI, bbox_inches='tight')
        plt.close()
    
    print(f"\n✓ All feature plots saved to: {figures_dir}")
    print(f"  - Batch plots: featureplot_markers_batch*.{FIGURE_FORMAT}")
    print(f"  - Category plots: featureplot_*.{FIGURE_FORMAT}")


Annotation Step 2b: Generating Marker Gene Feature Plots (UMAP)

Generating feature plots for 67 markers...
Creating 5 feature plot batch(es)...

  Batch 1/5: 16 markers
    ✓ Saved: featureplot_markers_batch1.png

  Batch 2/5: 16 markers
    ✓ Saved: featureplot_markers_batch2.png

  Batch 3/5: 16 markers
    ✓ Saved: featureplot_markers_batch3.png

  Batch 4/5: 16 markers
    ✓ Saved: featureplot_markers_batch4.png

  Batch 5/5: 3 markers
    ✓ Saved: featureplot_markers_batch5.png

  Generating category-specific feature plots...
    Basal: 4 markers
    Secretory: 3 markers
    Ciliated: 3 markers
    Goblet: 3 markers
    AT1: 3 markers
    AT2: 3 markers
    Proliferating: 3 markers

✓ All feature plots saved to: /home/h2048/data/py/1029/bbknn_celltype_analysis/Epithelial/subtype_analysis/figures_annotation
  - Batch plots: featureplot_markers_batch*.png
  - Category plots: featureplot_*.png
